In [ ]:
!pip install datasets==3.6.0
!pip install yandex-chain
!pip install transformers==4.28.1

  Using cached transformers-4.28.1-py3-none-any.whl.metadata (109 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.28.1-py3-none-any.whl (7.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
from datasets import load_dataset
import numpy
numpy.random.seed(42)

In [ ]:
dataset_train = load_dataset("AmazonScience/massive", "ru-RU", split='train')
dataset_validation = load_dataset("AmazonScience/massive", "ru-RU", split='validation')
dataset_test = load_dataset("AmazonScience/massive", "ru-RU", split='test')
print(dataset_train[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

massive.py: 0.00B [00:00, ?B/s]

ru-RU/train/0000.parquet:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/202k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/277k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11514 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2033 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2974 [00:00<?, ? examples/s]

{'id': '1', 'locale': 'ru-RU', 'partition': 'train', 'scenario': 16, 'intent': 48, 'utt': 'разбуди меня в девять утра в пятницу', 'annot_utt': 'разбуди меня в [time : девять утра] в [date : пятницу]', 'worker_id': '11', 'slot_method': {'slot': ['time', 'date'], 'method': ['translation', 'translation']}, 'judgments': {'worker_id': ['4', '32', '8'], 'intent_score': [1, 1, 1], 'slots_score': [1, 1, 1], 'grammar_score': [4, 4, 4], 'spelling_score': [2, 2, 2], 'language_identification': ['target', 'target', 'target']}}


In [ ]:
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score, adjusted_rand_score, calinski_harabasz_score, silhouette_score

def compute_metrics(points, true_labels, result_labels):
    ari = adjusted_rand_score(true_labels, result_labels)
    nmi = normalized_mutual_info_score(true_labels, result_labels)
    ami = adjusted_mutual_info_score(true_labels, result_labels)
    silhouette = silhouette_score(points, result_labels)
    ch = calinski_harabasz_score(points, result_labels)
    return [ari, nmi, ami, silhouette, ch]

def print_metrics(results):
    for result in reversed(sorted(results)):
        print(result[1])
        print(f"Adjusted Rand Index: {result[0][0]*100:.2f}")
        print(f"Normalized Mutual Information: {result[0][1]*100:.2f}")
        print(f"Adjusted Mutual Information: {result[0][2]*100:.2f}")
        print(f"Silhouette Score: {result[0][3]*100:.2f}")
        print(f"Calinski-Harabasz Score: {result[0][4]:.2f}")

Как классифицировать запросы, не привлекая внимание LLM? Требуется сделать две вещи: превратить тексты в вектора и как-то кластеризовать/классифицировать тексты, пользуясь векторами, полученными на предыдущем шаге.
В обоих случаях мы имеем несколько вариантов.

**Задача 1**. Векторизация.
1. Воспользуемся векторами сочетаний $n$ букв (например, 3).
2. Воспользуемся токенизаторами какой-нибудь существующей модели (например, ruBERT).

**Задача 2**. Классификация.
1. Воспользуемся существующими (чисто математическими) методами кластеризациями: метод $k$ средних, метод ближайшего соседа *etc*.
2. Обучение классификатора.

1-1. Вектор из сочетаний $n$ букв.

In [ ]:
n = 3
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, n), min_df=5)
txts = [item['utt'] for item in dataset_train]
vectorizer.fit(txts)

CountVectorizer(analyzer='char', min_df=5, ngram_range=(1, 3))

1-2. Токенизатор существующей модели.

In [ ]:
from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("ai-forever/FRIDA")

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

In [ ]:
letters = vectorizer.transform(txts)
berts = [tokenizer(txt)['input_ids'] for txt in txts]

Правда, некоторая проблема заключается в том, что в результате применения БЕРТа у векторов выходят разные дли́ны. Кластеризации методами вроде метода $k$ средних хотят единой длины. Проблема эта не нова.
Посмотрим для начала, сколько векторов какой длины есть.

In [ ]:
lens = {}
for elmnt in berts:
  length = len(elmnt)
  if length not in lens:
    lens.update([(length, 0)])
  lens[length] += 1
lens = sorted(list(lens.items()))

for pair in lens:
  print(*pair, sep="\t")

3	9
4	157
5	470
6	998
7	1329
8	1530
9	1477
10	1321
11	1009
12	821
13	650
14	437
15	355
16	274
17	179
18	146
19	97
20	67
21	55
22	40
23	27
24	21
25	15
26	7
27	10
28	3
29	3
31	1
32	2
34	2
35	1
39	1


Наш план — добавить в конце нулей.

In [ ]:
from tqdm import tqdm # эту ячейку приходится запускать дважды я не знаю почему

maxlen = max([len(element) for element in berts])
berts_new = []
for element in tqdm(berts):
  if len(element) < maxlen:
    zeros = [0]*(maxlen-len(element))
    elmnt = element.extend(zeros)
  else:
    elmnt = element
  berts_new.append(elmnt)

100%|██████████| 11514/11514 [00:00<00:00, 2318667.96it/s]


In [ ]:
print(berts_new)

[[1, 573, 296, 4309, 669, 282, 9382, 3733, 282, 9818, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1262, 546, 278, 571, 29950, 309, 1176, 3823, 3681, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 303, 326, 4203, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 370, 931, 24962, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 303, 326, 2201, 24962, 309, 4306, 6614, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 12087, 24962, 309, 4306, 6614, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 267, 1065, 295, 22615, 1233, 2513, 967, 22768, 6566, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 293, 28421, 34797, 1297, 23243, 527, 172

2-1. Кластеризация математическими методами.

2-1-1. Метод $k$ средних.

2-1-1 + 1-2. $k$ средних и токенизация БЕРТа.

In [ ]:
labels = len(set([obj['intent'] for obj in dataset_train]))
print(labels)

60


In [ ]:
from sklearn.cluster import KMeans

kmeans_ = KMeans(n_clusters=60, random_state=42).fit(berts_new)
labs = kmeans_.labels_

In [ ]:
# число кластеров найдено выше
results = []
best = 0
num = 0
for n in range(57,64):
  kmeans_ = KMeans(n_clusters=n, random_state=42).fit(berts_new)
  labs_b = kmeans_.labels_
  if n == 63: kmeans = kmeans_
  results.append((compute_metrics(berts_new, dataset_train['intent'], labs_b), n))
for result in reversed(sorted(results)):
    print(result[1])
    print(f"Adjusted Rand Index: {result[0][0]*100:.2f}")
    print(f"Normalized Mutual Information: {result[0][1]*100:.2f}")
    print(f"Adjusted Mutual Information: {result[0][2]*100:.2f}")
    print(f"Silhouette Score: {result[0][3]*100:.2f}")
    print(f"Calinski-Harabasz Score: {result[0][4]:.2f}")

63
Adjusted Rand Index: 1.55
Normalized Mutual Information: 10.88
Adjusted Mutual Information: 6.92
Silhouette Score: 15.60
Calinski-Harabasz Score: 439.87
61
Adjusted Rand Index: 1.52
Normalized Mutual Information: 10.57
Adjusted Mutual Information: 6.71
Silhouette Score: 15.68
Calinski-Harabasz Score: 447.68
62
Adjusted Rand Index: 1.49
Normalized Mutual Information: 10.58
Adjusted Mutual Information: 6.66
Silhouette Score: 15.59
Calinski-Harabasz Score: 442.73
60
Adjusted Rand Index: 1.36
Normalized Mutual Information: 10.37
Adjusted Mutual Information: 6.53
Silhouette Score: 15.76
Calinski-Harabasz Score: 447.79
59
Adjusted Rand Index: 1.34
Normalized Mutual Information: 10.25
Adjusted Mutual Information: 6.46
Silhouette Score: 15.51
Calinski-Harabasz Score: 450.02
58
Adjusted Rand Index: 1.34
Normalized Mutual Information: 10.23
Adjusted Mutual Information: 6.49
Silhouette Score: 15.51
Calinski-Harabasz Score: 455.26
57
Adjusted Rand Index: 1.33
Normalized Mutual Information: 10.1

Посмотрим для примера на какой-нибудь из кластеров.

In [ ]:
labs = kmeans.labels_
number = 42
for i, lab in enumerate(labs):
  if lab == number:
    print(txts[i])

запусти музыку из моей любимой станции pandora
закажи один кофе из кофейни starbucks
можешь пожалуйста заказать мне немного еды в mcdonalds
пожалуйста включи мой плейлист на spotify
могу я забрать свой большой заказ в pizza hut
мне нужно назначить встречу со светой завтра alexa
олли сколько звёзд получил спецназ на imdb
как сегодня обстоят дела с акциямиshell south
дай мне описание железного кулака на netflix
запости моё текущее местоположение в instagram
пожалуйста выложи это фото в мой instagram
твитни в аккаунт клиентской службы starbucks
найди сайт с формой для жалоб у starbucks
найди все входящие имейлы от amazon
у меня есть новый контакт новая почта собака gmail точка com написать письмо
есть ли новые непрочитанные письма в yahoo


Вообще говоря, некоторые запросы в этом кластере и правда сходны. Это неплохо, но, вообще говоря, скорее не видно некого объединяющего начала у кластеров.

Посмотрим, насколько вообще кластеры соответствуют реальным лейблам.

In [ ]:
data = [[0]*60 for i in range(0,61)]
for i, lab in enumerate(labs):
  true = dataset_train[i]['intent']
  data[lab][true] += 1

Безусловно, кластеры не обязаны соответствовать реальности. Но в любом случае хочется видеть какие-то скопления, а не равномерные распределения. Итак, выведем то, что получилось. В идеале каждый столбец и каждая строка должны иметь вид «одно не-нулевое значение и много нулей».

Напомним, что строки — предсказанные метки, столбцы — реальные.

In [ ]:
from tabulate import tabulate
print(tabulate(data))

---  --  --  --  --  --  --  -  --  --  --  --  ---  ---  --  -  --  --  -  --  --  --  --  --  --  --  --  -  -  -  --  --  ---  ---  --  --  --  -  -  --  --  -  --  --  --  ---  --  --  --  --  ---  --  --  --  --  --  --  --  --  --
  5   3   4   6   2   0   1  0   0  13  11  10   18    8   1  0   6   6  1  11   1   1   6   1   0   3  15  4  1  4  12   0   28    2   5   2  11  0  2   3   1  0   4   2  17   21   3   5   1  40   11   4   0  11   0   0   2   7   2  10
  0   1   2   2   1   0   1  0   0   2   1   1    6    1   1  0   4   2  0   3   0   2   7   0   0   2   2  1  1  0   7   0    5    1   1   0   1  0  0   0   1  0   4   0   7   11   1   3   0   5   10   0   1   0   0   1   0   1   5   5
  5   1   3   2   1   0   4  1   0   1   2   0    4   12   0  2   1   6  0   6  11   3   8   0   0   1   3  2  0  0   1   2   12    1   0   0  13  0  1   1   0  1   1   3   3   34   0   0   0   9    6   4   0   0   0   1   0   4  15   3
 15   5  13  10   5   0  11  0   0  22   7   7   34 

2-1-1 + 1-1. Метод $k$-средних и вектор сочетаний из 3 букв.

In [ ]:
# напомним, что у нас был список letters
kmeans_letter = KMeans(n_clusters=60).fit(letters)
labsL = kmeans_letter.labels_
number = 42
for i, lab in enumerate(labsL):
  if lab == number:
    print(txts[i])
dataL = [[0]*60 for i in range(0,61)]
for i, lab in enumerate(labsL):
  true = dataset_train[i]['intent']
  dataL[lab][true] += 1
print(tabulate(dataL))

время идти спать
олли время спать
olly приглуши свет на кухне
приглуши свет на кухне
пропылесось весь дом
сделай свет поярче
там сейчас идёт снег
перемешай этот плейлист
заставь меня рассмеяться
olly приглуши свет
сейчас я хочу кофе
я почти тебя не слышу олли
сделай свет ярко синим
сделай свет поярче
завтра обещают ветер
подсвети светильник
снег будет завтра
завтра будет тепло
там на улице сейчас тепло
поставь земфиру хочешь
играй песни светланы лободы
измени цвет света
вернуться к разговору после
отобрази текущее время
сохрани мой выбор музыки
продолжается ли лето
свари кофе в полдень
играй плейлист в случайном порядке
приглуши свет в гостиной
сохрани все песни нюши
играй плейлист группы звери
запусти последний альбом циклом
запусти все мои любимые песни
используй пылесос в холле
olly я хочу эспрессо
играй мой плейлист
играй мелодичную музыку
сделай свет красным
папа джонс есть в деливери
та песня моя любимая
привет olly как дела
играть песни светланы лободы
воспроизведи учебный
настр

NameError: name 'tabulate' is not defined

Здесь очень хорошо видно, что кластер №42 (во всяком случае, в том виде, в котором он был получен 25.02.2026 20:21) — это кластер запросов со словом «пожалуйста». Это неплохо. Но… думается, что не этим определяются интенты.

Напомним, что методов кластеризации существует много. Попробуем воспользоваться какими-то другими.

In [ ]:
from sklearn.cluster import SpectralClustering
spectrum_1 = SpectralClustering(n_clusters = 60)
spectrum_1.fit(berts_new)
# spectrum_2 = SpectralClustering(n_clusters = 60)
# spectrum_2.fit(letters)

/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (60). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


SpectralClustering(n_clusters=60)

Обнаружилось, что другие методы кластеризации с векторами CountVectorizer слишком ресурсозатратны (мы попробовали CountVectorizer с min_df=5, min_df=10, ngram_range = (1,2), ngram_range = (1,3), ngram_range = (3,3)).

In [ ]:
labs1 = spectrum_1.labels_
# number = 42
# for i, lab in enumerate(labs1):
#   if lab == number:
#     print(txts[i])
# data1 = [[0]*60 for i in range(0,61)]
# for i, lab in enumerate(labs1):
#   true = dataset_train[i]['intent']
#   data1[lab][true] += 1
# print(tabulate(data1))

In [ ]:
# labs2 = spectrum_2.labels_
# number = 42
# for i, lab in enumerate(labs2):
#   if lab == number:
#     print(txts[i])
# data2 = [[0]*60 for i in range(0,61)]
# for i, lab in enumerate(labs2):
#   true = dataset_train[i]['intent']
#   data2[lab][true] += 1
# print(tabulate(data2))

In [ ]:
from sklearn.cluster import AgglomerativeClustering
agc1 = AgglomerativeClustering(n_clusters = 60, linkage='ward')
agc1.fit(berts_new)
agc2 = AgglomerativeClustering(n_clusters = 60, linkage='ward')
agc2.fit(letters.toarray())

AgglomerativeClustering(n_clusters=60)

In [ ]:
from sklearn.cluster import HDBSCAN
hdbs1 = HDBSCAN()
hdbs1.fit(berts_new)
labs_h1 = hdbs1.labels_

In [ ]:
hdbs2 = HDBSCAN()
hdbs2.fit(letters.toarray())
labs_h2 = hdbs2.labels_

In [ ]:
labs3 = agc1.labels_
number = 42
for i, lab in enumerate(labs3):
  if lab == number:
    print(txts[i])
data3 = [[0]*60 for i in range(0,61)]
for i, lab in enumerate(labs3):
  true = dataset_train[i]['intent']
  data3[lab][true] += 1
print(tabulate(data3))

включи ивана дорна из моего плейлиста
воспроизвести последнюю песню из моего любимого плейлиста
покажи мне будильники которые я установил
включи пожалуйста сейчас на ужин классическую музыку
увеличь громкость так чтобы я мог слышать это в другой комнате
покажи мне самую хорошую погоду на этой неделе
включи ольгу бузову начиная с мало половин
покажи мне недельный прогноз для дома пожалуйста
включи я никому не верю после этой песни
скажи мне можно ли в евразии заказывать на вынос
включи моя попытка номер пять виа гры
собери все мои песни дани милохина
скажи сколько градусов было в нашем городе в двенадцать часов дня
включи музыку клавы коки без перемешивания
включи песни из в бой идут одни старики
воспроизведи домой и затем следующую новый мерин
будет хорошо если ты поменяешь свет на желтый
включи музыку сергея рахманинова
воспроизведи мне топ десять песен сплин этого года
мне нужно чтобы ты конвертировал девять утра центрального поясного времени в североамериканское восточное время
вклю

In [ ]:
labs4 = agc2.labels_
number = 42
for i, lab in enumerate(labs4):
  if lab == number:
    print(txts[i])
data4 = [[0]*60 for i in range(0,61)]
for i, lab in enumerate(labs4):
  true = dataset_train[i]['intent']
  data4[lab][true] += 1
print(tabulate(data4))

пожалуйста подскажи какое время будет в шесть вечера здесь в австралии
пусть робот пылесос убираеться с десяти и до одиннадцати утра каждый день
отфильтровать все грустные песни сектора газа и создать плейлист грустные песни сектора газа
переведи двадцать три тридцать из гринвича плюс четыре тридцать в два ноль ноль по гринвичу
переведи десять тридцать из времени по гринвичу плюс два тридцать в ноль ноль по гринвичу
у меня встреча сегодня в семь вечера выключи музыку на час в это время
я хочу поесть две шавермы в пите пожалуйста и диетическую колу
если я позвоню мише в новосибирск в шесть вечера который час будет у него
убедись в том что мои шаффлы всегда повторяются и всегда в случайном порядке
заказать саб с индейкой и оформить доставку на пять вечера сегодня
попроси их не забывать всегда запирать дверь и внимательно следить за своими ключами
пожалуйста скажи какой будет погода для города майкоп адыгея на следующей неделе
двадцать второе мая дата моего рождения и я хочу знать какой д

In [ ]:
from sklearn.cluster import BisectingKMeans
bkm1 = BisectingKMeans(n_clusters=60)
bkm1.fit(berts_new)
# bkm2 = BisectingKMeans(n_clusters=60)
# bkm2.fit(letters)

BisectingKMeans(n_clusters=60)

In [ ]:
labs5 = bkm1.labels_
number = 42
for i, lab in enumerate(labs5):
  if lab == number:
    print(txts[i])
data5 = [[0]*60 for i in range(0,61)]
for i, lab in enumerate(labs5):
  true = dataset_train[i]['intent']
  data5[lab][true] += 1
print(tabulate(data5))

время идти спать
олли время спать
там сейчас идёт снег
играй арию
что это за музыка
подними мне настроение
измени громкость
увеличь громкость
могу я заказать на вынос из испанского заведения
какое сегодня число
какой день недели сегодня
что это за песня
вымой пол пожалуйста
сегодня будет дождь
что там с погодой
запусти кофе машину
какой будет погода завтра
на какое время установлены мои будильники
что там с погодой сейчас
пойдет сегодня дождь
повторить альбом
сделай кофе
сделай красный свет в гостиной
я не способен услышать тебя не мог бы ты говорить немного громче
найди мне новости о речи трампа
какая температура снаружи
удаленный датчик
измени цвет света
насколько там холодно
сколько дней рождения приходится на двадцать третье
есть ли рождество двадцать второго
какая сейчас дата
увеличить громкость
в пять тридцать вечера завтра будет заход солнца
есть ли вероятность дождя на этой неделе
какой будет погода через неделю
поищи музыку госпел
отобрази текущее время
что там с погодой сегод

In [ ]:
# labs6 = bkm2.labels_
# number = 42
# for i, lab in enumerate(labs6):
#   if lab == number:
#     print(txts[i])
# data6 = [[0]*60 for i in range(0,61)]
# for i, lab in enumerate(labs6):
#   true = dataset_train[i]['intent']
#   data6[lab][true] += 1
# print(tabulate(data6))

In [ ]:
all_labs = {"BERTKMeans": labs, "CountVecKMeans": labsL, "BERTSpectral": labs1, "BERTAGC": labs3, "CountVecAGC": labs4, "BERTBKMeans": labs5, "BERTHDBSCAN": labs_h1, "CountVecHDBSCAN": labs_h2}
all_metrics = []
for key, value in all_labs.items():
    if key in ("CountVecKMeans", "CountVecAGC", "CountVecHDBSCAN"):
        all_metrics.append((compute_metrics(letters.toarray(), dataset_train['intent'], value), key))
    else:
        all_metrics.append((compute_metrics(berts_new, dataset_train['intent'], value), key))
print_metrics(all_metrics)
print()
print(f"BERTHDBSCAN cluster count: {len(set(labs_h1))}")
print(f"CountVecHDBSCAN cluster count: {len(set(labs_h2))}")

CountVecAGC
Adjusted Rand Index: 8.17
Normalized Mutual Information: 33.98
Adjusted Mutual Information: 31.11
Silhouette Score: -2.62
Calinski-Harabasz Score: 69.60
CountVecKMeans
Adjusted Rand Index: 6.87
Normalized Mutual Information: 28.13
Adjusted Mutual Information: 25.02
Silhouette Score: -0.54
Calinski-Harabasz Score: 87.14
BERTKMeans
Adjusted Rand Index: 1.55
Normalized Mutual Information: 10.88
Adjusted Mutual Information: 6.92
Silhouette Score: 15.60
Calinski-Harabasz Score: 439.87
BERTBKMeans
Adjusted Rand Index: 1.31
Normalized Mutual Information: 10.20
Adjusted Mutual Information: 6.27
Silhouette Score: 8.33
Calinski-Harabasz Score: 319.79
BERTAGC
Adjusted Rand Index: 1.15
Normalized Mutual Information: 9.86
Adjusted Mutual Information: 6.01
Silhouette Score: 12.45
Calinski-Harabasz Score: 377.56
BERTHDBSCAN
Adjusted Rand Index: 0.35
Normalized Mutual Information: 15.65
Adjusted Mutual Information: 6.03
Silhouette Score: -46.01
Calinski-Harabasz Score: 9.76
CountVecHDBSCAN

In [ ]:
# hdbs3 = HDBSCAN(min_cluster_size=10)
# hdbs3.fit(berts_new)
# hdbs4 = HDBSCAN(min_cluster_size=10)
# hdbs4.fit(letters.toarray())
# hdbs5 = HDBSCAN(min_cluster_size=15)
# hdbs5.fit(berts_new)
# hdbs6 = HDBSCAN(min_cluster_size=15)
# hdbs6.fit(letters.toarray())
# labs_h3 = hdbs3.labels_
# labs_h4 = hdbs4.labels_
# labs_h5 = hdbs5.labels_
# labs_h6 = hdbs6.labels_
# hdbs_labs = {"HDBSBmin10": labs_h3, "HDBSCmin10": labs_h4, "HDBSBmin15": labs_h5, "HDBSCmin15": labs_h6}
# hdbs_metrics = []

# for key, value in hdbs_labs.items():
#     hdbs_metrics.append((compute_metrics(berts_new, dataset_train['intent'], value), key))
# print_metrics(hdbs_metrics)
# print()
# print(f"HDBSBmin10 cluster count: {len(set(labs_h3))}")
# print(f"HDBSCmin10 cluster count: {len(set(labs_h4))}")
# print(f"HDBSBmin15 cluster count: {len(set(labs_h5))}")
# print(f"HDBSCmin15 cluster count: {len(set(labs_h6))}")

In [ ]:
# from sklearn.cluster import HDBSCAN
# hdbs7 = HDBSCAN(min_cluster_size=7)
# hdbs7.fit(berts_new)
# hdbs8 = HDBSCAN(min_cluster_size=7)
# hdbs8.fit(letters.toarray())
# labs_h7 = hdbs7.labels_
# labs_h8 = hdbs8.labels_
# hdbs_labs = {"HDBSBmin7": labs_h7, "HDBSCmin7": labs_h8}
# hdbs_metrics = []

# for key, value in hdbs_labs.items():
#     hdbs_metrics.append((compute_metrics(berts_new, dataset_train['intent'], value), key))
# print_metrics(hdbs_metrics)
# print()
# print(f"HDBSBmin7 cluster count: {len(set(labs_h7))}")
# print(f"HDBSCmin7 cluster count: {len(set(labs_h8))}")

In [ ]:
# from sklearn.cluster import HDBSCAN
# hdbss = HDBSCAN(min_cluster_size=5)
# hdbss.fit(berts_new)

In [ ]:
import pickle
# It is important to use binary access
# with open('hbbscan.pickle', 'wb') as f:
#     pickle.dump(hdbss, f)
# from google.colab import files
# files.download('hbbscan.pickle')

In [ ]:
with open('hbbscan.pickle', 'wb') as f:
    pickle.dump(agc1, f)
from google.colab import files
files.download('hbbscan.pickle')

with open('kmeans_count', 'wb') as f:
    pickle.dump(kmeans_letter, f)
from google.colab import files
files.download('hbbscan.pickle')

with open('kmeans_frida.pickle', 'wb') as f:
    pickle.dump(kmeans, f)
from google.colab import files
files.download('kmeans_frida.pickle')

with open('bkm_frida.pickle', 'wb') as f:
    pickle.dump(bkm1, f)
from google.colab import files
files.download('bkm_frida.pickle')

with open('agc_frida.pickle', 'wb') as f:
    pickle.dump(agc1, f)
from google.colab import files
files.download('agc_frida.pickle')

GPT-решение. Zero-shot.

In [ ]:
!curl --request POST --data '{"yandexPassportOauthToken":""}' https://iam.api.cloud.yandex.net/iam/v1/tokens
#  мой обычный

In [ ]:
iamToken = ""
folder_id = ""

In [ ]:
import json
from yandex_chain import YandexLLM, YandexGPTModel
LLM = YandexLLM(model=YandexGPTModel.Pro, folder_id=folder_id, iam_token=iamToken)

In [ ]:
from tqdm import tqdm

def zero_shot(LLM, data):
  labels = []
  possible_labels = []
  for element in tqdm(data):
    ss = f'Вот набор существующих намерений: {'\n'.join(possible_labels)}. ---' if len(possible_labels) > 0 else ''
    prompt = f'Твоя задача — определить намерение пользователя, произнесшего некоторое предложение. Это намерение может быть из списка существующих намерений или быть новым, сформулированным тобой на основании текста запроса и общих знаний. Выбери намерение из набора существующих намерений или назови новую, если ни одна из существующих не подходит. Тебе НЕЛЬЗЯ использовать в своём ответе слово «намерение» ни при каких обстоятельствах. Слово «новое» не используй. Твой ответ должен быть коротким (5 или меньше слов). \n{ss}.\nВот предложение, намерение которого нужно определить:\n{element}'
    # print(prompt)
    result = LLM(prompt)
    res = ' '.join(result.split()[:5])
    labels.append(res)
    if res not in possible_labels:
      possible_labels.append(res)
  return(labels)
LLM = YandexLLM(model=YandexGPTModel.Pro, folder_id=folder_id, iam_token=iamToken)
# gpt_labs = []
# possible_labs = []
# subset_of_txts = txts[:1000]
# labels = zero_shot(LLM, subset_of_txts)
# print(labels)

In [ ]:
print(len(list(set(labels))))

GPT-решение. Few-shot.
Попробовать добавлять в промпт пример на каждый существующий интент? Тогда промпт получится довольно длинным.

In [ ]:
def few_shot(LLM, data):
  labels = []
  possible_labels = []
  few_shot_examples = """<поставь будильник на два часа вперёд> Тема: {поставить будильник}\n
  <подними мне настроение> Тема: {шутка}\n
  <включи мой любимый плейлист> Тема: {включить музыку}\n
  <какой прогноз погоды на завтра> Тема: {прогноз погоды}\n
  <эта песня на заднем фоне очень раздражающая> Тема: {не нравится музыка}"""
  for element in tqdm(data):
    ss = f'Вот набор существующих намерений: {'\n'.join(possible_labels)}. ---' if len(possible_labels) > 0 else ''
    prompt = f"""Твоя задача — определить намерение пользователя, произнесшего некоторое предложение.
    Это намерение может быть из списка существующих намерений или быть новым, сформулированным тобой на основании текста запроса и общих знаний.
    Выбери намерение из набора существующих намерений или назови новую, если ни одна из существующих не подходит.
    Тебе НЕЛЬЗЯ использовать в своём ответе слово «намерение» ни при каких обстоятельствах. Слово «тема» также не нужно использовать. Слово «новое» не используй.
    Твой ответ должен быть коротким (5 или меньше слов). \n{ss}.Также ниже приведено несколько примеров предложений и соответствующих им тем.
    Каждое предложение заключено в лапки <>, а после слова Тема с двоеточием идет его тема в фигурных скобках {{}}.\n{few_shot_examples}\n
    Вот предложение, намерение которого нужно определить:\n{element}"""
    # print(prompt)
    result = LLM(prompt)
    res = ' '.join(result.split()[:5])
    labels.append(res)
    if res not in possible_labels:
      possible_labels.append(res)
  return(labels)
LLM = YandexLLM(model=YandexGPTModel.Pro, folder_id=folder_id, iam_token=iamToken)
# gpt_labs = []
# possible_labs = []
# labels_few = few_shot(LLM, subset_of_txts)
# print(labels_few)
# print(len(list(set(labels_few))))

In [ ]:
print(len(set(dataset_train['intent'][:1000])))

In [ ]:
metrics_1 = compute_metrics(berts_new[:1000], dataset_train['intent'][:1000], labels)
metrics_1 = (metrics_1, 'zero')
metrics_2 = compute_metrics(berts_new[:1000], dataset_train['intent'][:1000], labels_few)
metrics_2 = (metrics_2, 'few')
gpts = [metrics_1, metrics_2]
print_metrics(gpts)

10-шот

In [ ]:
from tqdm import tqdm

def ten_shot(LLM, data):
  labels = []
  possible_labels = []
  few_shot_examples = """<поставь будильник на два часа вперёд> Тема: {поставить будильник}\n
  <подними мне настроение> Тема: {шутка}\n
  <включи мой любимый плейлист> Тема: {включить музыку}\n
  <какой прогноз погоды на завтра> Тема: {прогноз погоды}\n
  <эта песня на заднем фоне очень раздражающая> Тема: {не нравится музыка}\n
  <расскажи мне последние новости> Тема: {новости}\n
  <какой день новогодняя ночь в этом году> Тема: {вопрос о дате}\n
  <сделай мне кофе без подсластителя> Тема: {сделать кофе}\n
  <как дела с заказом еды> Тема: {вопрос о заказе еды}\n
  <из какого фильма музыка как она называется> Тема: {вопрос о музыке}
  """
  for element in tqdm(data):
    ss = f'Вот набор существующих намерений: {'\n'.join(possible_labels)}. ---' if len(possible_labels) > 0 else ''
    prompt = f"""Твоя задача — определить намерение пользователя, произнесшего некоторое предложение.
    Это намерение может быть из списка существующих намерений или быть новым, сформулированным тобой на основании текста запроса и общих знаний.
    Выбери намерение из набора существующих намерений или назови новую, если ни одна из существующих не подходит.
    Тебе НЕЛЬЗЯ использовать в своём ответе слово «намерение» ни при каких обстоятельствах. Слово «тема» также не нужно использовать. Слово «новое» не используй.
    Твой ответ должен быть коротким (5 или меньше слов). \n{ss}.Также ниже приведено несколько примеров предложений и соответствующих им тем.
    Каждое предложение заключено в лапки <>, а после слова Тема с двоеточием идет его тема в фигурных скобках {{}}.\n{few_shot_examples}\n
    Вот предложение, намерение которого нужно определить:\n{element}"""
    # print(prompt)
    result = LLM(prompt)
    res = ' '.join(result.split()[:5])
    labels.append(res)
    if res not in possible_labels:
      possible_labels.append(res)
  return(labels)
LLM = YandexLLM(model=YandexGPTModel.Pro, folder_id=folder_id, iam_token=iamToken)
# gpt_labs = []
# subset_of_txts = txts[:1000]
# possible_labs = []
# labels_ten = ten_shot(LLM, subset_of_txts)

In [ ]:
print(len(list(set(labels_ten))))
metrics_3 = compute_metrics(berts_new[:1000], dataset_train['intent'][:1000], labels_ten)
print(metrics_3)
metrics_3 = (metrics_3, 'few_10')
gpts_3 = [metrics_3]
print_metrics(gpts_3)

In [ ]:
# сравним три метода
dataset_eval = load_dataset("AmazonScience/massive", "ru-RU", split='validation')
eval_txts = [item['utt'] for item in dataset_eval]
eval_intents = [item['intent'] for item in dataset_eval]
eval_berts = [tokenizer(txt)['input_ids'] for txt in eval_txts]

maxlen = max([len(element) for element in eval_berts])
berts_new_eval = []
for element in tqdm(berts):
  if len(element) < maxlen:
    zeros = [0]*(maxlen-len(element))
    elmnt = element.extend(zeros)
  else:
    elmnt = element
  berts_new_eval.append(elmnt)

100%|██████████| 11514/11514 [00:00<00:00, 2610444.12it/s]


In [ ]:
print(len(list(set(eval_intents[:1000]))))

39


Банк примеров

In [ ]:
import pandas as pd
url=f'https://docs.google.com/spreadsheet/ccc?key=1xSxkJ8IpczJSWfxju-HUrS9qE5wKIsBkHNZWfVYmDXg&output=xlsx'
dataframe = pd.read_excel(url,sheet_name='Лист1')
eng = dataframe['English']
rus = dataframe['Russian']
translate = {}
for i, element in enumerate(eng):
  translate.update([(element, rus[i])])
print(translate)

{'calendar_set': 'поставить календарь', 'play_music': 'включить музыку', 'alarm_remove': 'выключить будильник', 'play_audiobook': 'поставить аудиокнигу', 'iot_hue_lightdim': 'уменьшить свет', 'datetime_query': 'вопрос о дате', 'audio_volume_up': 'увеличить громкость', 'cooking_recipe': 'кулинарный рецепт', 'email_sendemail': 'послать письмо', 'qa_factoid': 'вопрос о фактах', 'transport_ticket': 'билет на транспорт', 'lists_createoradd': 'создать список', 'play_podcasts': 'поставить подкаст', 'weather_query': 'вопрос о погоде', 'recommendation_locations': 'порекомендовать места', 'music_query': 'вопрос о музыке', 'news_query': 'вопрос о новостях', 'alarm_set': 'поставить будильник', 'general_quirky': 'общий вопрос', 'calendar_remove': 'удалить календарь', 'recommendation_movies': 'порекомендовать фильм', 'transport_taxi': 'заказ такси', 'email_query': 'вопрос о письме', 'iot_coffee': 'заказ кофе', 'lists_remove': 'удалить список', 'alarm_query': 'вопрос о будильнике', 'lists_query': 'во

In [ ]:
from datasets import Dataset
def modif(string, translation=None):
    if translate:
      return translate[string]
    else:
      return string
def example_bank(data, size, translation=None):
  bank = {}
  data = Dataset.shuffle(data)#, seed=45)
  for elmnt in data:
    text, intent = elmnt['utt'], modif(dataset_test.features["intent"].int2str(elmnt['intent']))
    if intent not in bank.values():
      bank.update([(text, intent)])
    if len(bank.keys()) == size:
      break
  return bank
def modifier(bank):
  prompt_fragment = ""
  for key, value in bank.items():
    prompt_fragment += f"<{key}> Тема:{{{value}}}\n"
  return prompt_fragment[:-1]

bnk = example_bank(dataset_test, 10, translation=translate)
print(modifier(bnk))
for key, value in bnk.items():
  print(key, value, sep="\t")

<включи аудио сначала> Тема:{поставить аудиокнигу}
<пожалуйста закажи билет на поезд из екатеринбурга в нижний тагил на завтрашнее утро> Тема:{билет на транспорт}
<пришли мне напоминание о встрече с виталием в следующую пятницу> Тема:{поставить календарь}
<поменяй весь свет в доме на голубой> Тема:{изменить свет}
<включи мой перетасованный плейлист> Тема:{включить музыку}
<включи танки онлайн> Тема:{сыграть в игру}
<есть какие нибудь новости про выборы> Тема:{вопрос о новостях}
<давай твитнем жалобу> Тема:{вопрос о соцсетях}
<мои контакты в большинстве своем мужчины или женщины> Тема:{вопрос о контактах}
<выключи свет в гостиной> Тема:{выключить свет}
включи аудио сначала	поставить аудиокнигу
пожалуйста закажи билет на поезд из екатеринбурга в нижний тагил на завтрашнее утро	билет на транспорт
пришли мне напоминание о встрече с виталием в следующую пятницу	поставить календарь
поменяй весь свет в доме на голубой	изменить свет
включи мой перетасованный плейлист	включить музыку
включи тан

In [ ]:
def twenty_shot(LLM, data):
  labels = []
  possible_labels = []
  few_shot_examples = """<поставь будильник на два часа вперёд> Тема: {поставить будильник}\n
  <подними мне настроение> Тема: {шутка}\n
  <включи мой любимый плейлист> Тема: {включить музыку}\n
  <какой прогноз погоды на завтра> Тема: {прогноз погоды}\n
  <эта песня на заднем фоне очень раздражающая> Тема: {не нравится музыка}\n
  <расскажи мне последние новости> Тема: {новости}\n
  <какой день новогодняя ночь в этом году> Тема: {вопрос о дате}\n
  <сделай мне кофе без подсластителя> Тема: {сделать кофе}\n
  <как дела с заказом еды> Тема: {вопрос о заказе еды}\n
  <из какого фильма музыка как она называется> Тема: {вопрос о музыке}\n
  <пожалуйста проверь темы популярные в twitter> Тема:{вопрос о соцсетях}\n
  <удалить будильник> Тема:{выключить будильник}\n
  <привет не могла ли ты приглушить свет пожалуйста> Тема:{уменьшить свет}\n
  <как приготовить индейку> Тема:{рецепт}
  <когда отсюда уезжает следующий поезд в город> Тема:{вопрос о транспорте}\n
  <пожалуйста закажи билет на поезд из екатеринбурга в нижний тагил на завтрашнее утро> Тема:{билет на транспорт}\n
  <мне нравится эта музыка можешь ли ты пожалуйста сохринить её в мой танцевальный плейлист и запомнить что она мне нравиться> Тема:{нравится музыка}\n
  <назови все текущие события в моем родном городе> Тема:{порекомендовать мероприятие}\n
  <назови мне лучшие туристические места для посещения в америке> Тема:{порекомендовать места}\n
  <удали пункт из списка> Тема:{удалить список}\n
  """
  for element in tqdm(data):
    ss = f'Вот набор существующих намерений: {'\n'.join(possible_labels)}. ---' if len(possible_labels) > 0 else ''
    prompt = f"""Твоя задача — определить намерение пользователя, произнесшего некоторое предложение.
    Это намерение может быть из списка существующих намерений или быть новым, сформулированным тобой на основании текста запроса и общих знаний.
    Выбери намерение из набора существующих намерений или назови новую, если ни одна из существующих не подходит.
    Тебе НЕЛЬЗЯ использовать в своём ответе слово «намерение» ни при каких обстоятельствах. Слово «тема» также не нужно использовать. Слово «новое» не используй.
    Твой ответ должен быть коротким (5 или меньше слов). \n{ss}.Также ниже приведено несколько примеров предложений и соответствующих им тем.
    Каждое предложение заключено в лапки <>, а после слова Тема с двоеточием идет его тема в фигурных скобках {{}}.\n{few_shot_examples}\n
    Вот предложение, намерение которого нужно определить:\n{element}"""
    # print(prompt)
    result = LLM(prompt)
    res = ' '.join(result.split()[:5])
    labels.append(res)
    if res not in possible_labels:
      possible_labels.append(res)
  return(labels)
LLM = YandexLLM(model=YandexGPTModel.Pro, folder_id=folder_id, iam_token=iamToken)
# gpt_labs = []
# possible_labs = []
# labels_twenty = twenty_shot(LLM, eval_subset)
# metrs_4 = compute_metrics(berts_new_eval[:1000], eval_intents[:1000], labels_twenty)
# metrs_4 = (metrs_4, 'twenty')
# metrs = [metrs_4]
# print_metrics(metrs)

In [ ]:
print(len(list(set(labels_twenty))))

87


In [ ]:
for i in range(0,250):
  print(txts[i])
  print(labels[i])
  print(labels_few[i])
  number = dataset_train['intent'][i]
  intent = dataset_train.features["intent"].int2str(number)
  print(intent)
  print()

In [ ]:
eval_subset = eval_txts[:1000]
eval_zero = zero_shot(LLM, eval_subset)
print(len(list(set(eval_zero))))
eval_five = few_shot(LLM, eval_subset)
print(len(list(set(eval_five))))
eval_ten = ten_shot(LLM, eval_subset)
print(len(list(set(eval_ten))))
eval_twenty = twenty_shot(LLM, eval_subset)
print(len(list(set(eval_twenty))))

  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_5488/2315922401.py:10: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use invoke instead.
  result = LLM(prompt)
100%|██████████| 1000/1000 [07:12<00:00,  2.31it/s]


175


100%|██████████| 1000/1000 [07:18<00:00,  2.28it/s]


124


100%|██████████| 1000/1000 [07:25<00:00,  2.24it/s]


81


100%|██████████| 1000/1000 [07:38<00:00,  2.18it/s]

109


TypeError: write() argument must be str, not list

In [ ]:
with open('0.txt', 'w') as f:
  for elmnt in eval_zero:
    f.write(f'{elmnt}\n')
with open('5.txt', 'w') as f:
  for elmnt in eval_five:
    f.write(f'{elmnt}\n')
with open('10.txt', 'w') as f:
  for elmnt in eval_ten:
    f.write(f'{elmnt}\n')
with open('20.txt', 'w') as f:
  for elmnt in eval_twenty:
    f.write(f'{elmnt}\n')

In [ ]:
from google.colab import files
files.download('0.txt')
files.download('5.txt')
files.download('10.txt')
files.download('20.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
metrs_1 = compute_metrics(berts_new_eval[:1000], eval_intents[:1000], eval_zero)
metrs_1 = (metrs_1, 'zero')
metrs_2 = compute_metrics(berts_new_eval[:1000], eval_intents[:1000], eval_five)
metrs_2 = (metrs_2, 'five')
metrs_3 = compute_metrics(berts_new_eval[:1000], eval_intents[:1000], eval_ten)
metrs_3 = (metrs_3, 'ten')
metrs_4 = compute_metrics(berts_new_eval[:1000], eval_intents[:1000], eval_twenty)
metrs_4 = (metrs_4, 'twenty')
gpts = [metrs_1, metrs_2, metrs_3, metrs_4]
print_metrics(gpts)

ten
Adjusted Rand Index: 62.77
Normalized Mutual Information: 75.55
Adjusted Mutual Information: 68.95
Silhouette Score: -51.66
Calinski-Harabasz Score: 0.90
twenty
Adjusted Rand Index: 53.09
Normalized Mutual Information: 76.58
Adjusted Mutual Information: 69.17
Silhouette Score: -57.75
Calinski-Harabasz Score: 0.92
zero
Adjusted Rand Index: 52.06
Normalized Mutual Information: 73.58
Adjusted Mutual Information: 62.26
Silhouette Score: -61.36
Calinski-Harabasz Score: 0.97
five
Adjusted Rand Index: 47.92
Normalized Mutual Information: 73.82
Adjusted Mutual Information: 65.12
Silhouette Score: -59.37
Calinski-Harabasz Score: 1.03


In [ ]:
for i in range(0,500):
  print(eval_subset[i])
  print(eval_zero[i])
  print(eval_five[i])
  print(eval_ten[i])
  print(eval_twenty[i])
  number = eval_intents[i]
  intent = dataset_train.features["intent"].int2str(number)
  print(intent)
  print()

выключи свет пожалуйста
Выключить свет.
{выключить свет}
{выключить свет}
{выключить свет}
iot_hue_lightoff

приглуши свет в гостиной
Изменить яркость света
{приглушить свет}
{приглушить свет}
{уменьшить свет}
iot_hue_lightdim

сделай в комнате темнее
Изменить яркость света.
{приглушить свет}
{приглушить свет}
{уменьшить свет}
iot_hue_lightdim

убери квартиру
Убраться в квартире
{уборка квартиры}
{убраться в квартире}
{уборка}
iot_cleaning

уборка это хорошо пыль это так плохо сделай сейчас свою магию почисти мой ковёр
Убраться в квартире
{уборка квартиры}
{убраться в квартире}
{уборка}
iot_cleaning

покажи статус доступной памяти
Проверить статус памяти
{узнать статус памяти}
{узнать статус памяти}
{узнать статус памяти}
general_quirky

назови самые популярные сервисы доставки китайской еды
Узнать популярные сервисы доставки
{узнать популярные сервисы доставки еды}
{вопрос о сервисах доставки}
{порекомендовать сервисы доставки еды}
takeaway_query

найди мне тайскую еду на вынос около 

In [ ]:
for i in range(500,1000):
  print(eval_subset[i])
  print(eval_zero[i])
  print(eval_five[i])
  print(eval_ten[i])
  print(eval_twenty[i])
  number = eval_intents[i]
  intent = dataset_train.features["intent"].int2str(number)
  print(intent)
  print()

я хочу послушать отличную музыку
Прослушать музыку.
{включить музыку}
{включить музыку}
{включить музыку}
play_music

увеличить яркость лампы
Изменить яркость света
{приглушить свет или звук}
{изменить настройки освещения}
{увеличить свет}
iot_hue_lightup

пожалуйста поставь громкость на самую громкую настройку
Изменить уровень громкости.
{приглушить свет или звук}
{изменить настройки воспроизведения музыки}
{увеличить громкость}
audio_volume_up

убери звук аудио
Изменить уровень громкости.
{приглушить свет или звук}
{приглушить звук} или {выключить устройство}
{выключить звук}
audio_volume_mute

останови звук
Выключить устройство/изменить режим устройства.
{выключить устройство} или {приглушить свет
{выключить устройство} или {приглушить звук}
{выключить звук}
audio_volume_mute

хоть на этой неделе будет дождь
Узнать прогноз погоды
{узнать прогноз погоды}
{прогноз погоды}
{прогноз погоды}
weather_query

приглуши весь свет внутри
Изменить яркость света
{приглушить свет}
{приглушить све

In [ ]:
with open('truth.txt', 'w') as f:
  for i in range(0,1000):
    number = eval_intents[i]
    intent = dataset_train.features["intent"].int2str(number)
    f.write(f'{intent}\n')
files.download('truth.txt')
with open('texts.txt', 'w') as f:
  for i in range(0,1000):
    f.write(f'{eval_subset[i]}\n')
files.download('texts.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2000 примеров из трейна для итогового обучения

In [ ]:
final = Dataset.shuffle(dataset_train, seed=610)
final_texts = [item['utt'] for item in final]
final_intents = [dataset_test.features["intent"].int2str(elmnt['intent']) for elmnt in final]
final_berts = [tokenizer(txt)['input_ids'] for txt in final_texts]
from tqdm import tqdm # эту ячейку приходится запускать дважды я не знаю почему

maxlen_final = max([len(element) for element in final_berts])
berts_final = []
for element in tqdm(final_berts):
  if len(element) < maxlen_final:
    zeros = [0]*(maxlen_final-len(element))
    elmnt = element.extend(zeros)
  else:
    elmnt = element
  berts_final.append(elmnt)

size = 2000
fintext_subset = final_texts[:size]
finint_subset = final_intents[:size]
finbert_subset = final_berts[:size]

100%|██████████| 11514/11514 [00:00<00:00, 236200.00it/s]


In [ ]:
print(fintext_subset)
print(finint_subset)
print(finbert_subset)

['страница в википедии о сергее безрукове', 'вернись к острову сокровищ', 'включи какое нибудь радио', 'олли что посмотреть на этих выходных', 'воспроизведи покинула чат', 'перечисли информацию о книжных ярмарках в москве на следующей неделе', 'сегодня я ходил за покупками и купил огромный дилдо', 'пожалуйста включи песню мало половин', 'покажи мне несколько новостей т.а.с.с.', 'каково время сеансов мажора', 'ты можешь гарантировать что ввыберешь выигрышные лотерейные номера', 'какой день недели сегодня', 'какое напоминание я поставил на завтра', 'удали все события в моём календаре', 'заказать поездку в бар', 'письма полученые с десяти вечера до семи утра не должны остаться без ответа', 'я хочу чтобы ты выключил розетку до того как телефон зарядится', 'удали моё последнее мероприятие', 'покажи номер и контактный адрес электронной почты рината', 'сколько контактов с именем женя', 'на каком уровне находятся акции яндекса', 'опубликуй на facebook я дома', 'воспроизведи азазель бориса акун

Формулировка семантики кластеров.

In [ ]:
def cluster_finder(labels, texts):
  length = len(list(set(labels)))
  result = [[] for _ in range(length)]
  for i, text in enumerate(texts):
    result[labels[i]].append(text)
  return result

In [ ]:
print(len(kmeans.labels_))
print(len(txts))

In [ ]:
km_cl = cluster_finder(kmeans.labels_, txts)
hd_cl = cluster_finder(hdbss.labels_, txts)

In [ ]:
from tqdm import tqdm
def cluster_semantics_request(LLM, clusters):
  answers = []
  for l in tqdm(clusters):
    data = '\n'.join(l)
    answer = LLM(f'Ты — эксперт по определению намерений пользователя. Ниже представлено несколько предложений-запросов к чат-боту. Каждое отдельное предложение записано в лапках: <>. Очень коротко назови тематику или намерение, который объединяет все эти запросы. Твой ответ должен содержать не более 5 слов. Ответ не должен содержать слова «намерение».\n{data}')
    answer_abbr = ' '.join(answer.split()[:5])
    answers.append(answer_abbr)
  return answers

In [ ]:
print(cluster_semantics_request(LLM, km_cl[:20]))
print(cluster_semantics_request(LLM, hd_cl[:20]))